# 01 — Explorer et contrôler les données

Ce notebook lit par défaut la démonstration **synthétique**, à créer avec `python -m src.demo`. Pour le projet, remplacer explicitement le chemin par le manifeste réel vérifié. Aucun chiffre de démonstration ne doit devenir un résultat CAMELYON17.

Objectif : expliquer les unités patch/lame/groupe patient, les effectifs et les contrôles avant entraînement.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

# Se placer à la racine rend les chemins relatifs du manifeste cohérents.
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from src.data.audit import read_manifest, audit

# Modifier cette ligne pour un lot réel ; ne pas mélanger les deux sources.
MANIFEST = ROOT / "outputs/demo_synthetic/manifest.csv"
if not MANIFEST.is_file():
    raise FileNotFoundError("Exécuter la démonstration ou choisir un manifeste réel existant.")
frame = read_manifest(MANIFEST)
synthetic = "source" in frame and (frame.source == "synthetic").all()
display(Markdown("**SYNTHÉTIQUE : aucune conclusion médicale.**" if synthetic else "**Données fournies : vérifier provenance et annotations.**"))
frame.head()

## 1. Séparer avant de mesurer

L’audit refuse les labels inconnus, les chemins dupliqués, les patientes/lames partagées entre ensembles et les centres OOD incohérents. Un audit réussi ne prouve pas l’exactitude des annotations ni l’absence de copies renommées. Les groupes patient CAMELYON17 sont artificiels.

In [ ]:
print(audit(frame, check_files=True))
print("Groupes patient :", frame.patient_id.nunique())
print("Lames :", frame.slide_id.nunique())
print("Patchs :", len(frame))

## 2. Compter sans double-compter les patientes

Un même groupe peut contribuer aux deux classes. Ne pas sommer les nombres de patientes par classe comme s’il s’agissait de groupes disjoints. Les étiquettes de test ne servent pas à régler le modèle.

In [ ]:
counts = frame.groupby(["split", "center_id", "label"]).agg(
    patches=("path", "size"), patients=("patient_id", "nunique"), slides=("slide_id", "nunique")
).reset_index()
display(counts)
plot_data = counts.pivot_table(index=["split", "center_id"], columns="label", values="patches", aggfunc="sum", fill_value=0)
ax = plot_data.plot.bar(stacked=True, figsize=(9, 4), title=("SYNTHÉTIQUE — " if synthetic else "") + "Effectifs")
ax.set_ylabel("Nombre de patchs");

## 3. Voir des exemples d’entraînement

Choix déterministe : un exemple de chaque classe dans train. Pour des données réelles, commenter résolution, tissu, qualité et annotation. Ne pas chercher une interprétation histologique dans le bruit artificiel.

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for label, ax in zip(("0", "1"), axes):
    group = frame[(frame.split == "train") & (frame.label == label)].sort_values("path")
    ax.axis("off")
    if group.empty:
        ax.set_title(f"Classe {label} absente")
        continue
    row = group.iloc[0]
    with PILImage.open(row.path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"Classe {label} — centre {row.center_id}")
fig.suptitle("SYNTHÉTIQUE" if synthetic else "Exemples de train");

## 4. Interprétation à rédiger

- Quelle source et quelle définition du label avons-nous réellement ?
- Combien de groupes et de centres indépendants couvrons-nous ?
- Quelles limites peut créer le pilote réduit ?
- Les classes sont-elles équilibrées et cela reflète-t-il le contexte clinique ?
- Quelles vérifications sur les annotations restent nécessaires ?

**Sortie attendue :** un tableau d’effectifs, des exemples lisibles et un commentaire argumenté. Aucun entraînement n’est lancé automatiquement par ce notebook.